# Step 2 – Post-optimisation simulations

`N_SIMULATIONS` simulations per farm and metric at the optimum from step 1 (lowest re-evaluated loss across runs), followed for `POST_HORIZON_DAYS`. Run step 1 first.

**Outputs** (`results/post_simulations/`):

| File | Content |
| --- | --- |
| `parameters_used.csv` | Optimum used per farm and metric |
| `case_incidence.csv`, `case_incidence_summary.csv` | Simulated vs observed daily cases (Fig. 4) |
| `herd_daily.csv`, `herd_daily_summary.csv` | Daily S, E, I, R, N, new exposures, new infectious, βH (Fig. 3, Fig. 2B) |
| `pen_daily_summary.csv` | Daily states and new infections per pen (second wave) |
| `infected_by_lactation.csv` | Infections by pen of exposure and lactation group |
| `sim_summary.csv`, `sim_summary_stats.csv` | Per run: peaks, timing, totals, attack rate, losses, reproduction numbers (Fig. 2A) |

In [1]:
import time
import warnings

import numpy as np
import pandas as pd

import ab_sim as m
import common as c
import config as cfg

from workers import post_simulation_task

warnings.filterwarnings("ignore", category=RuntimeWarning)   # empty-window means

## Functions

In [2]:
def pen_summary(arrs: list, new_I_arrs: list, farm: str, metric: str) -> pd.DataFrame:
    """Mean, median and 2.5/97.5 percentiles across simulations per day, pen and state
    (S, E, I, R, new infectious)."""
    A = np.stack(arrs)                 # sim, day, pen, state
    B = np.stack(new_I_arrs)           # sim, day, pen
    rows = []
    days = np.arange(1, A.shape[1] + 1)
    for p in range(m.N_PENS):
        series = {m.STATE_NAMES[s_]: A[:, :, p, s_] for s_ in range(4)}
        series["new_infectious"] = B[:, :, p]
        for var, X in series.items():
            q = np.percentile(X, [2.5, 50, 97.5], axis=0)
            rows.append(pd.DataFrame({"farm": farm, "metric": metric, "day": days, "pen": p + 1,
                                      "pen_name": m.PEN_NAMES[p + 1], "variable": var,
                                      "mean": X.mean(0), "median": q[1], "p2_5": q[0], "p97_5": q[2]}))
    return pd.concat(rows)

In [3]:
def daily_summary(df: pd.DataFrame, by: list, cols: list) -> pd.DataFrame:
    """Mean, median and 2.5/97.5 percentiles of each column within groups (long format)."""
    g = df.groupby(by)
    parts = []
    for col in cols:
        s = g[col].agg(mean="mean", median="median",
                       p2_5=lambda x: x.quantile(0.025), p97_5=lambda x: x.quantile(0.975))
        s.insert(0, "variable", col)
        parts.append(s.reset_index())
    return pd.concat(parts)

## Optima and simulations

In [4]:
c.ensure_dirs()
OUT = c.DIRS["post_simulations"]
farms = m.load_inputs(cfg)
c.farm_burn_ins(farms, verbose=False)

opt = pd.read_csv(c.DIRS["optimization"] / "optimal_parameters.csv")
best = opt.loc[opt.groupby(["farm", "metric"])["loss"].idxmin()]
best.to_csv(OUT / "parameters_used.csv", index=False)
best

,farm,metric,run,T,R,I,IC,loss,loss_optimiser,infectious_period_days,duration_of_infection_days
0,Farm1,DM,0,0.001571,0.066749,1,1,213.181729,212.832264,14.981559,15.981559
1,Farm1,PLF,0,0.001571,0.066749,1,1,286.766172,222.665880,14.981559,15.981559
2,Farm2,DM,0,0.002210,0.079428,1,1,83.266364,74.070510,12.590053,13.590053
3,Farm2,PLF,0,0.000978,0.078151,1,8,181.341928,187.710372,12.795788,13.795788
4,Farm3,DM,0,0.002640,0.052711,1,5,204.497431,252.713439,18.971355,19.971355
5,Farm3,PLF,0,0.003757,0.077983,1,3,415.184024,379.797631,12.823373,13.823373
6,Farm4,DM,0,0.000577,0.079753,2,8,352.021465,319.734848,12.538764,14.538764
7,Farm4,PLF,0,0.000577,0.079753,2,8,237.881713,232.548211,12.538764,14.538764


In [5]:
tasks = []
for r in best.itertuples():
    params = {"T": float(r.T), "R": float(r.R), "I": int(r.I), "IC": int(r.IC)}
    tasks += [(r.farm, r.metric, s, params) for s in range(cfg.N_SIMULATIONS)]

t0 = time.time()
print(f"Post-simulations: {len(tasks)} runs of {cfg.POST_HORIZON_DAYS} days")
results = c.pool_map(post_simulation_task, tasks, chunksize=4)
print(f"done in {time.time() - t0:.0f} s")

Post-simulations: 800 runs of 365 days
done in 11 s


## Save per-simulation data

In [6]:
cases = pd.concat([r[0] for r in results])
herd = pd.concat([r[1] for r in results])
summ = pd.DataFrame([r[4] for r in results])
lact = pd.concat([r[5] for r in results])

cases.to_csv(OUT / "case_incidence.csv", index=False)
herd.to_csv(OUT / "herd_daily.csv", index=False)
summ.to_csv(OUT / "sim_summary.csv", index=False)
lact.to_csv(OUT / "infected_by_lactation.csv", index=False)

## Summaries

In [7]:
cs = daily_summary(cases, ["farm", "metric", "day"], ["simulated"])
cs = cs.merge(cases.groupby(["farm", "metric", "day"])["observed"].first().reset_index(),
              on=["farm", "metric", "day"])
cs.to_csv(OUT / "case_incidence_summary.csv", index=False)

daily_summary(herd, ["farm", "metric", "day"],
              ["S", "E", "I", "R", "N", "new_exposed", "new_infectious", "beta_H"]
              ).to_csv(OUT / "herd_daily_summary.csv", index=False)

In [8]:
pens, groups = [], {}
for (farm, metric, _, _), r in zip(tasks, results):
    groups.setdefault((farm, metric), ([], []))
    groups[(farm, metric)][0].append(r[2])
    groups[(farm, metric)][1].append(r[3])
for (farm, metric), (a, b) in groups.items():
    pens.append(pen_summary(a, b, farm, metric))
pd.concat(pens).to_csv(OUT / "pen_daily_summary.csv", index=False)

In [9]:
stat_cols = [col for col in summ.columns
             if col not in ("farm", "metric", "sim") + tuple(c.PARAMS)]
stats = c.summarise(summ, ["farm", "metric"], stat_cols)
stats.to_csv(OUT / "sim_summary_stats.csv", index=False)
stats[stats.variable.str.startswith(("R_", "R0_H", "attack"))]

,farm,metric,variable,mean,sd,median,p2_5,p97_5,n
0,Farm1,DM,R0_H_d1_10,4.586926,3.743596,4.134379,0.000000,12.909041,100
1,Farm1,DM,R0_H_d1_15,4.225518,3.165504,4.557678,0.000000,10.052350,100
2,Farm1,DM,R0_H_d1_peak,4.225518,3.165504,4.557678,0.000000,10.052350,100
3,Farm1,DM,R_cohort_d1_10,2.235963,1.555946,2.500000,0.000000,5.175000,100
4,Farm1,DM,R_cohort_d1_15,1.838247,1.183561,2.282075,0.000000,3.507576,100
...,...,...,...,...,...,...,...,...,...
206,Farm4,PLF,R_cohort_d1_10,3.524337,0.402340,3.559373,2.731154,4.333654,100
207,Farm4,PLF,R_cohort_d1_15,2.957044,0.301957,2.933986,2.478022,3.604407,100
208,Farm4,PLF,R_cohort_d1_peak,3.410476,0.346555,3.444444,2.763194,4.050500,100
209,Farm4,PLF,R_index_cows,3.592500,1.038042,3.500000,1.809375,5.506250,100


In [10]:
r = stats[stats.variable == "R0_H_d1_15"]
r.pivot(index="farm", columns="metric", values=["median", "p2_5", "p97_5"]).round(2)

median         p2_5        p97_5       
metric     DM    PLF    DM   PLF     DM    PLF
farm                                          
Farm1    4.56   3.04  0.00  0.00  10.05  11.14
Farm2    2.50   1.32  0.00  0.33   7.69   2.17
Farm3   13.82  12.31  7.75  6.91  15.56  14.60
Farm4    6.17   6.16  3.88  3.80   7.74   7.55

In [11]:
cols = ["T", "R", "I", "IC", "R_index_cows", "R_cohort_d1_10", "R_cohort_d1_15",
        "R0_H_d1_15", "attack_rate_horizon", "cumulative_cases_fit_window"]
check = (summ.groupby(["farm", "metric"])[cols].median()
             .join(pd.Series({k: f["observed"].sum() for k, f in c.FARMS.items()}
                             if c.FARMS else {k: f["observed"].sum() for k, f in m.load_inputs(cfg).items()},
                             name="observed_cases"), on="farm"))
check.round(3)

T      R    I   IC  R_index_cows  R_cohort_d1_10  \
farm  metric                                                         
Farm1 DM      0.002  0.067  1.0  1.0         3.000           2.500   
      PLF     0.002  0.067  1.0  1.0         2.000           2.050   
Farm2 DM      0.002  0.079  1.0  1.0         2.000           1.550   
      PLF     0.001  0.078  1.0  8.0         1.062           1.101   
Farm3 DM      0.003  0.053  1.0  5.0         5.100           1.736   
      PLF     0.004  0.078  1.0  3.0         5.333           1.415   
Farm4 DM      0.001  0.080  2.0  8.0         3.750           3.545   
      PLF     0.001  0.080  2.0  8.0         3.500           3.559   

              R_cohort_d1_15  R0_H_d1_15  attack_rate_horizon  \
farm  metric                                                    
Farm1 DM               2.282       4.558                0.478   
      PLF              2.012       3.040                0.481   
Farm2 DM               1.609       2.503                0.229   
      PLF              1.125       1.317                0.199   
Farm3 DM               1.315      13.820                0.917   
      PLF              1.226      12.307                0.749   
Farm4 DM               2.966       6.166                0.820   
      PLF              2.934       6.159                0.814   

              cumulative_cases_fit_window  observed_cases  
farm  metric                                               
Farm1 DM                             84.5           135.0  
      PLF                            34.0           135.0  
Farm2 DM                             42.5            70.0  
      PLF                            56.5            70.0  
Farm3 DM                            646.0           605.0  
      PLF                           634.5           605.0  
Farm4 DM                            282.5           269.0  
      PLF                           264.0           269.0

In [12]:
# R0: secondary exposures caused by ONE index cow in a fully susceptible herd,
# simulated with each farm's optimal T, R and I
burns = c.farm_burn_ins(farms, verbose=False)
r0_days = 150      # long enough for the index cow's infectious period to end
rows = []
for r in best.itertuples():
    for s in range(cfg.N_SIMULATIONS):
        rng = np.random.default_rng(m.task_seed("r0_single", r.farm, r.metric, s))
        res = m.simulate(burns[r.farm], r.T, r.R, int(r.I), 1, r0_days, rng,
                         cfg.SEED_PENS, cfg.MASKED_PENS, cfg.ANNUAL_CULL_RATE, keep_infected=True)
        inf = res.infected
        rows.append({"farm": r.farm, "metric": r.metric, "sim": s,
                     "index_pen": int(inf.loc[inf.is_index, "inf_pen"].iloc[0]),
                     "R0_single_index": int(inf.loc[inf.is_index, "secondary"].iloc[0])})
r0 = pd.DataFrame(rows)
r0.to_csv(OUT / "R0_single_index.csv", index=False)

r0_stats = (r0.groupby(["farm", "metric"])["R0_single_index"]
              .agg(mean="mean", median="median",
                   p2_5=lambda x: x.quantile(0.025), p97_5=lambda x: x.quantile(0.975),
                   share_no_spread=lambda x: (x == 0).mean())
              .reset_index())
r0_stats.to_csv(OUT / "R0_single_index_stats.csv", index=False)
r0_stats.round(2)

,farm,metric,mean,median,p2_5,p97_5,share_no_spread
0,Farm1,DM,3.56,3.0,0.0,10.00,0.12
1,Farm1,PLF,2.55,2.0,0.0,9.52,0.24
2,Farm2,DM,2.75,1.0,0.0,10.00,0.31
3,Farm2,PLF,1.16,1.0,0.0,6.00,0.49
4,Farm3,DM,6.30,7.0,0.0,13.00,0.10
5,Farm3,PLF,5.73,5.0,0.0,14.52,0.07
6,Farm4,DM,4.60,4.0,0.0,14.00,0.16
7,Farm4,PLF,3.53,2.0,0.0,12.05,0.22
